# 01 · Demand Forecasting — beating the naive baseline
Per-day wine demand from `wine_consumption_log` (live if `SUPABASE_URL`/`SUPABASE_SERVICE_ROLE_KEY` are set, synthetic otherwise).

**The discipline:** every model must beat the *seasonal-naive* forecast (same weekday last week) on **MASE** — this mirrors `engine/forecasting.ts` in the API, so notebook findings transfer 1:1 into production parameters.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from wineops_data import get_checks, get_tables, get_consumption, get_orders, get_inventory, daily_series
plt.rcParams['figure.figsize'] = (11, 4)

cons = get_consumption()
y = daily_series(cons, 'created_at', 'quantity')
y.plot(title=f'Daily bottles ({len(y)} days)'); plt.show()

In [ ]:
# Train/test split: last 28 days held out
H = 28
train, test = y.iloc[:-H], y.iloc[-H:]

def mase(actual, pred, train, m=7):
    naive = np.mean(np.abs(train[m:].values - train[:-m].values))
    return np.mean(np.abs(actual - pred)) / naive

snaive = y.shift(7).iloc[-H:]  # seasonal naive baseline
print('baseline MASE (seasonal naive) = 1.00 by construction:', round(mase(test.values, snaive.values, train.values), 3))

In [ ]:
# Holt-Winters (statsmodels) — the production model's big sibling
from statsmodels.tsa.holtwinters import ExponentialSmoothing
hw = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=7).fit()
hw_pred = hw.forecast(H)
print('Holt-Winters MASE:', round(mase(test.values, hw_pred.values, train.values), 3))

In [ ]:
# SARIMA — captures autocorrelation HW misses
from statsmodels.tsa.statespace.sarimax import SARIMAX
sar = SARIMAX(train, order=(1,0,1), seasonal_order=(1,1,1,7)).fit(disp=False)
sar_pred = sar.forecast(H)
print('SARIMA MASE:', round(mase(test.values, sar_pred.values, train.values), 3))

In [ ]:
# Gradient boosting with calendar features — the ML route
from sklearn.ensemble import HistGradientBoostingRegressor
def feats(idx):
    return pd.DataFrame({'dow': idx.dayofweek, 'dom': idx.day, 'week': idx.isocalendar().week.astype(int),
                         'month': idx.month, 't': np.arange(len(idx))}, index=idx)
Xtr, Xte = feats(train.index), feats(test.index)
Xte['t'] += len(train)
gbm = HistGradientBoostingRegressor(max_iter=300).fit(Xtr, train.values)
gbm_pred = gbm.predict(Xte)
print('GBM MASE:', round(mase(test.values, gbm_pred, train.values), 3))

In [ ]:
ax = test.plot(label='actual', lw=2)
pd.Series(hw_pred.values, index=test.index).plot(ax=ax, label='Holt-Winters')
pd.Series(sar_pred.values, index=test.index).plot(ax=ax, label='SARIMA')
pd.Series(gbm_pred, index=test.index).plot(ax=ax, label='GBM')
ax.legend(); ax.set_title('28-day holdout'); plt.show()

**Transfer to production:** if SARIMA/GBM consistently beat Holt-Winters by >10% MASE here, promote the winner into the Python orchestrator; otherwise keep the in-process Holt-Winters (`/analytics/forecast`) — simpler and explainable. Re-run monthly as data accumulates.